## Random Forest pe CIC-IDS2017

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, accuracy_score)

daten_pfad = Path('data/Wednesday-WorkingHours.csv')
ergebnis_ord = Path('results')
ergebnis_ord.mkdir(exist_ok=True)

seeds = [1, 2, 3]
test_size = 0.3
n_baum = 100

### 1. Daten einlesen und bereinigen

In [2]:
df = pd.read_csv(daten_pfad, low_memory=False)
df.columns = df.columns.str.strip()
print('Zeilen, Spalten (roh):', df.shape)

label = [c for c in df.columns if c.lower() == 'label'][0]
df[label] = df[label].astype(str).str.strip()
df[label].value_counts()

Zeilen, Spalten (roh): (496779, 84)


Label
BENIGN                          319216
DoS Hulk                        158469
DoS GoldenEye                     7567
DoS slowloris                     4001
DoS Slowhttptest - Attempted      3369
DoS Slowhttptest                  1742
DoS slowloris - Attempted         1731
DoS Hulk - Attempted               593
DoS GoldenEye - Attempted           80
Heartbleed                          11
Name: count, dtype: int64

In [3]:
# attempted flows entfernen
ist_attempted = df[label].str.contains('attempted', case=False)
print('Entfernte Attempted-Zeilen:', ist_attempted.sum())
df = df[~ist_attempted].copy()

Entfernte Attempted-Zeilen: 5773


In [4]:
id_spalte = ['id', 'Flow ID', 'Src IP', 'Source IP', 'Dst IP', 'Destination IP',
              'Src Port', 'Source Port', 'Timestamp', 'Attempted Category']
entfernt = [c for c in df.columns if c in id_spalte]
print('Entfernte Spalten:', entfernt)
df = df.drop(columns=entfernt)

features = df.drop(columns=[label]).select_dtypes(include='number').columns.tolist()
df = df[features + [label]]
print('Anzahl Features:', len(features))

port_spalte = next((c for c in features if c in ['Dst Port', 'Destination Port']), None)
print('Port-Spalte:', port_spalte)

Entfernte Spalten: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Timestamp']
Anzahl Features: 78
Port-Spalte: Dst Port


In [5]:
df = df.replace([np.inf, -np.inf], np.nan)
print('Zeilen mit fehlenden/unendlichen Werten:', df.isna().any(axis=1).sum())
df = df.dropna()

vorher = len(df)
df = df.drop_duplicates()                                  # komplett identische Zeilen
nach_identisch = len(df)
df = df[~df.duplicated(subset=features, keep=False)]       # gleiche Features, unterschiedliches Label
print(f'Zeilen vorher: {vorher} | nach identischen Duplikaten: {nach_identisch} | '
      f'nach widersprüchlichen Duplikaten: {len(df)}')

konstant = [c for c in features if df[c].nunique() <= 1]
features = [c for c in features if c not in konstant]
print('Entfernte konstante Spalten:', konstant)
print('Finale Anzahl Features:', len(features))

Zeilen mit fehlenden/unendlichen Werten: 30
Zeilen vorher: 490976 | nach identischen Duplikaten: 472029 | nach widersprüchlichen Duplikaten: 472029
Entfernte konstante Spalten: ['Fwd URG Flags', 'Bwd URG Flags', 'URG Flag Count']
Finale Anzahl Features: 75


In [6]:
X = df[features]
y = (df[label].str.upper() != 'BENIGN').astype(int)
etiketten = df[label]

print(y.value_counts().rename({0: 'BENIGN (0)', 1: 'ANGRIFF (1)'}))
print()
print(etiketten.value_counts())

Label
BENIGN (0)     300655
ANGRIFF (1)    171374
Name: count, dtype: int64

Label
BENIGN              300655
DoS Hulk            158469
DoS GoldenEye         7567
DoS slowloris         3596
DoS Slowhttptest      1731
Heartbleed              11
Name: count, dtype: int64


### Hauptexperiment

In [7]:
def bewerte(y_wahr, y_vorh):
    tn, fp, fn, tp = confusion_matrix(y_wahr, y_vorh, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        'Recall': recall_score(y_wahr, y_vorh, zero_division=0),
        'Precision': precision_score(y_wahr, y_vorh, zero_division=0),
        'F1': f1_score(y_wahr, y_vorh, zero_division=0),
        'Accuracy': accuracy_score(y_wahr, y_vorh),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    }


def mittel_std(tabelle, spalten):
    return tabelle[spalten].agg(['mean', 'std']).T

In [8]:
splits = {}     
modelle = {}
ergebnisse = []

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed)
    splits[seed] = (X_train, X_test, y_train, y_test)

    rf = RandomForestClassifier(n_estimators=n_baum, n_jobs=-1, random_state=seed)
    t0 = time.perf_counter()
    rf.fit(X_train, y_train)
    t_training = time.perf_counter() - t0

    y_vorh = rf.predict(X_test)
    modelle[seed] = rf

    r = bewerte(y_test, y_vorh)
    r.update({'Seed': seed, 'Trainingszeit_s': t_training})
    ergebnisse.append(r)
    print(f"Seed {seed}: FPR={r['FPR']:.5f}  Recall={r['Recall']:.5f}  "
          f"Precision={r['Precision']:.5f}  F1={r['F1']:.5f}  Training={t_training:.1f}s")

haupt = pd.DataFrame(ergebnisse)
haupt.to_csv(ergebnis_ord / 'rf_hauptexperiment.csv', index=False)
haupt[['Seed', 'TN', 'FP', 'FN', 'TP']]

Seed 1: FPR=0.00001  Recall=0.99990  Precision=0.99998  F1=0.99994  Training=13.0s
Seed 2: FPR=0.00000  Recall=0.99982  Precision=1.00000  F1=0.99991  Training=11.5s
Seed 3: FPR=0.00002  Recall=0.99998  Precision=0.99996  F1=0.99997  Training=11.8s


,Seed,TN,FP,FN,TP
0,1,90196,1,5,51407
1,2,90197,0,9,51403
2,3,90195,2,1,51411


### Mittelwert ± Standardabweichung über die Seeds

In [9]:
mittel_std(haupt, ['FPR', 'Recall', 'Precision', 'F1', 'Accuracy', 'Trainingszeit_s'])

,mean,std
FPR,0.000011,0.000011
Recall,0.999903,0.000078
Precision,0.999981,0.000019
F1,0.999942,0.000029
Accuracy,0.999958,0.000021
Trainingszeit_s,12.072021,0.776298


### Geschwindigkeit


In [10]:
n_latenz = 1000


def messe_geschwindigkeit(rf, X_test, n=n_latenz):
    # 1. alle Test-Flows in einem Aufruf
    t0 = time.perf_counter()
    rf.predict(X_test)
    batch_ms = 1000 * (time.perf_counter() - t0) / len(X_test)

    # 2. Einzel-Flow: ein Flow pro Aufruf, wie ein Alarm im SOC
    rf.n_jobs = 1                      # bei einer Zeile kostet Parallelisierung mehr, als sie bringt
    flows = X_test.sample(n, random_state=0)
    zeiten = []
    for i in range(n):
        t0 = time.perf_counter()
        rf.predict(flows.iloc[[i]])
        zeiten.append(1000 * (time.perf_counter() - t0))
    rf.n_jobs = -1

    return batch_ms, np.median(zeiten), np.percentile(zeiten, 95)


geschw = []
for seed in seeds:
    _, X_test, _, _ = splits[seed]
    batch_ms, median_ms, p95_ms = messe_geschwindigkeit(modelle[seed], X_test)
    geschw.append({'Seed': seed, 'Batch_ms_pro_Flow': batch_ms,
                   'Latenz_Median_ms': median_ms, 'Latenz_P95_ms': p95_ms})
    print(f'Seed {seed}: Batch {batch_ms:.5f} ms/Flow | Einzel-Flow Median {median_ms:.2f} ms, P95 {p95_ms:.2f} ms')

geschw = pd.DataFrame(geschw)
geschw.to_csv(ergebnis_ord / 'rf_geschwindigkeit.csv', index=False)
geschw[['Batch_ms_pro_Flow', 'Latenz_Median_ms', 'Latenz_P95_ms']].agg(['mean', 'std']).T

Seed 1: Batch 0.00088 ms/Flow | Einzel-Flow Median 5.64 ms, P95 6.66 ms
Seed 2: Batch 0.00216 ms/Flow | Einzel-Flow Median 6.04 ms, P95 7.90 ms
Seed 3: Batch 0.00128 ms/Flow | Einzel-Flow Median 5.75 ms, P95 7.61 ms


,mean,std
Batch_ms_pro_Flow,0.001439,0.000653
Latenz_Median_ms,5.808888,0.205493
Latenz_P95_ms,7.390403,0.646586


### feature importance

In [11]:
importance = pd.DataFrame({seed: modelle[seed].feature_importances_ for seed in seeds},
                          index=features)
importance['Mittelwert'] = importance[seeds].mean(axis=1)
importance['Std'] = importance[seeds].std(axis=1)
importance = importance.sort_values('Mittelwert', ascending=False)
importance.to_csv(ergebnis_ord / 'rf_feature_importance.csv')

if port_spalte:
    rang = importance.index.get_loc(port_spalte) + 1
    print(f'Rang von {port_spalte}: {rang} von {len(features)}')
importance[['Mittelwert', 'Std']].head(15)

Rang von Dst Port: 23 von 75


,Mittelwert,Std
Packet Length Variance,0.113481,0.022932
Packet Length Std,0.105778,0.023121
Bwd Packet Length Std,0.101953,0.045455
Bwd Packet Length Mean,0.089891,0.033329
Average Packet Size,0.069141,0.015799
Bwd Packet Length Max,0.060023,0.020269
Packet Length Max,0.058761,0.010399
Subflow Bwd Bytes,0.052481,0.015109
Packet Length Mean,0.047350,0.020696
Bwd Segment Size Avg,0.044196,0.022043


### Einfluss der "Destination Port"

In [12]:
assert port_spalte is not None, 'Keine Destination-Port-Spalte gefunden'
features_ohne_port = [c for c in features if c != port_spalte]

# Wie verteilt sich der Port
for klasse, name in [(0, 'BENIGN'), (1, 'ANGRIFF')]:
    anteile = X.loc[y == klasse, port_spalte].value_counts(normalize=True).head(5).round(4)
    print(f'{name}: häufigste Ports (Anteil)')
    print(anteile.to_string(), '\n')

BENIGN: häufigste Ports (Anteil)
Dst Port
53     0.5944
443    0.2550
80     0.1143
123    0.0127
22     0.0035 

ANGRIFF: häufigste Ports (Anteil)
Dst Port
80     0.9999
444    0.0001 



In [13]:
ablation = []
for seed in seeds:
    X_train, X_test, y_train, y_test = splits[seed]
    rf_ohne = RandomForestClassifier(n_estimators=n_baum, n_jobs=-1, random_state=seed)
    rf_ohne.fit(X_train[features_ohne_port], y_train)
    r = bewerte(y_test, rf_ohne.predict(X_test[features_ohne_port]))
    r.update({'Seed': seed, 'Variante': 'ohne Port'})
    ablation.append(r)

    r_mit = haupt.loc[haupt['Seed'] == seed].iloc[0].to_dict()
    r_mit['Variante'] = 'mit Port'
    ablation.append(r_mit)

ablation = pd.DataFrame(ablation)
ablation.to_csv(ergebnis_ord / 'rf_ablation_port.csv', index=False)
ablation.groupby('Variante')[['FPR', 'Recall', 'Precision', 'F1']].agg(['mean', 'std']).round(5)

FPR           Recall          Precision                F1  \
              mean      std    mean      std      mean      std     mean   
Variante                                                                   
mit Port   0.00001  0.00001  0.9999  0.00008   0.99998  0.00002  0.99994   
ohne Port  0.00001  0.00001  0.9999  0.00007   0.99998  0.00002  0.99994   

                    
               std  
Variante            
mit Port   0.00003  
ohne Port  0.00002

### LOAO - leave one attack out

In [14]:
angriffstypen = etiketten[etiketten.str.upper() != 'BENIGN'].value_counts()
angriffstypen = angriffstypen[angriffstypen >= 50].index.tolist()
print('LOAO-Angriffstypen:', angriffstypen)

benign_index = etiketten[etiketten.str.upper() == 'BENIGN'].index


def loao_lauf(spalten, unbekannt, seed):
    b_train, b_test = train_test_split(benign_index, test_size=test_size, random_state=seed)
    bekannt = etiketten[(etiketten.str.upper() != 'BENIGN') & (etiketten != unbekannt)].index
    unbekannt_idx = etiketten[etiketten == unbekannt].index

    train_idx = b_train.union(bekannt)
    test_idx = b_test.union(unbekannt_idx)

    rf_l = RandomForestClassifier(n_estimators=n_baum, n_jobs=-1, random_state=seed)
    t0 = time.perf_counter()
    rf_l.fit(X.loc[train_idx, spalten], y.loc[train_idx])
    t_training = time.perf_counter() - t0

    vorh = pd.Series(rf_l.predict(X.loc[test_idx, spalten]), index=test_idx)
    return {
        'Unbekannter_Angriff': unbekannt,
        'Seed': seed,
        'Flows_unbekannt': len(unbekannt_idx),
        'Recall_unbekannt': vorh.loc[unbekannt_idx].mean(),
        'FPR': vorh.loc[b_test].mean(),
        'F1': f1_score(y.loc[test_idx], vorh, zero_division=0),
        'Trainingszeit_s': t_training,
    }


loao = []
for variante, spalten in [('mit Port', features), ('ohne Port', features_ohne_port)]:
    for typ in angriffstypen:
        for seed in seeds:
            r = loao_lauf(spalten, typ, seed)
            r['Variante'] = variante
            loao.append(r)
        letzte = pd.DataFrame(loao[-len(seeds):])
        print(f"{variante:9s} | {typ:18s} Recall_unbekannt = "
              f"{letzte['Recall_unbekannt'].mean():.4f} ± {letzte['Recall_unbekannt'].std():.4f}")

loao = pd.DataFrame(loao)
loao.to_csv(ergebnis_ord / 'rf_loao_alle_laeufe.csv', index=False)

LOAO-Angriffstypen: ['DoS Hulk', 'DoS GoldenEye', 'DoS slowloris', 'DoS Slowhttptest']
mit Port  | DoS Hulk           Recall_unbekannt = 0.2408 ± 0.1571
mit Port  | DoS GoldenEye      Recall_unbekannt = 0.7014 ± 0.0080
mit Port  | DoS slowloris      Recall_unbekannt = 0.5026 ± 0.0028
mit Port  | DoS Slowhttptest   Recall_unbekannt = 0.8344 ± 0.0569
ohne Port | DoS Hulk           Recall_unbekannt = 0.1573 ± 0.0606
ohne Port | DoS GoldenEye      Recall_unbekannt = 0.6924 ± 0.0177
ohne Port | DoS slowloris      Recall_unbekannt = 0.4972 ± 0.0006
ohne Port | DoS Slowhttptest   Recall_unbekannt = 0.8257 ± 0.0151


In [15]:
loao_zusammenfassung = (loao
    .groupby(['Unbekannter_Angriff', 'Variante'])[['Flows_unbekannt', 'Recall_unbekannt', 'FPR', 'F1']]
    .agg({'Flows_unbekannt': 'first',
          'Recall_unbekannt': ['mean', 'std'],
          'FPR': ['mean', 'std'],
          'F1': ['mean', 'std']})
    .round(4))
loao_zusammenfassung.to_csv(ergebnis_ord / 'rf_loao_zusammenfassung.csv')
loao_zusammenfassung

Flows_unbekannt Recall_unbekannt          FPR  \
                                        first             mean     std mean   
Unbekannter_Angriff Variante                                                  
DoS GoldenEye       mit Port             7567           0.7014  0.0080  0.0   
                    ohne Port            7567           0.6924  0.0177  0.0   
DoS Hulk            mit Port           158469           0.2408  0.1571  0.0   
                    ohne Port          158469           0.1573  0.0606  0.0   
DoS Slowhttptest    mit Port             1731           0.8344  0.0569  0.0   
                    ohne Port            1731           0.8257  0.0151  0.0   
DoS slowloris       mit Port             3596           0.5026  0.0028  0.0   
                    ohne Port            3596           0.4972  0.0006  0.0   

                                        F1          
                               std    mean     std  
Unbekannter_Angriff Variante                        
DoS GoldenEye       mit Port   0.0  0.8244  0.0055  
                    ohne Port  0.0  0.8182  0.0125  
DoS Hulk            mit Port   0.0  0.3707  0.2078  
                    ohne Port  0.0  0.2686  0.0922  
DoS Slowhttptest    mit Port   0.0  0.9085  0.0340  
                    ohne Port  0.0  0.9040  0.0085  
DoS slowloris       mit Port   0.0  0.6687  0.0024  
                    ohne Port  0.0  0.6641  0.0004

### Stichprobe fur die LLMs

In [16]:
stichprob_seed = 1
n_klasse = 100

_, X_test_s, _, y_test_s = splits[stichprob_seed]
rng = np.random.default_rng(stichprob_seed)

benign_test = y_test_s[y_test_s == 0].index
angriff_test = y_test_s[y_test_s == 1].index

auswahl_benign = rng.choice(benign_test, size=n_klasse, replace=False)

typen_test = etiketten.loc[angriff_test]
quote = n_klasse // typen_test.nunique()
auswahl_angriff = []
for typ, gruppe in typen_test.groupby(typen_test):
    k = min(quote, len(gruppe))
    auswahl_angriff.extend(rng.choice(gruppe.index, size=k, replace=False))
rest = np.setdiff1d(angriff_test, auswahl_angriff)
auswahl_angriff.extend(rng.choice(rest, size=n_klasse - len(auswahl_angriff), replace=False))

stichprobe_idx = np.concatenate([auswahl_benign, np.array(auswahl_angriff)])
stichprobe = df.loc[stichprobe_idx, features + [label]].copy()
stichprobe['y'] = y.loc[stichprobe_idx].values
stichprobe['RF_Vorhersage'] = modelle[stichprob_seed].predict(stichprobe[features])
stichprobe = stichprobe.sample(frac=1, random_state=stichprob_seed)   # Reihenfolge mischen
stichprobe.index.name = 'Original_Index'
stichprobe.to_csv(ergebnis_ord / 'llm_stichprobe_200.csv')

print(stichprobe[label].value_counts())

Label
BENIGN              100
DoS Hulk             35
DoS GoldenEye        21
DoS slowloris        21
DoS Slowhttptest     20
Heartbleed            3
Name: count, dtype: int64


### Beispiele für Few-Shot-Prompts

In [17]:
X_train_s, _, y_train_s, _ = splits[stichprob_seed]
rng_fs = np.random.default_rng(stichprob_seed)

fs_benign = rng_fs.choice(y_train_s[y_train_s == 0].index, size=4, replace=False).tolist()
typen_train = etiketten.loc[y_train_s[y_train_s == 1].index]
fs_angriff = [rng_fs.choice(gruppe.index) for _, gruppe in typen_train.groupby(typen_train)]

fs_idx = fs_benign + fs_angriff
few_shot = df.loc[fs_idx, features + [label]].copy()
few_shot['y'] = y.loc[fs_idx].values
few_shot = few_shot.sample(frac=1, random_state=stichprob_seed)
few_shot.index.name = 'Original_Index'
few_shot.to_csv(ergebnis_ord / 'llm_fewshot_beispiele.csv')

assert not set(few_shot.index) & set(stichprobe.index), 'Few-Shot-Beispiele überschneiden sich mit der Stichprobe!'
few_shot[[label]]

,Label
Original_Index,
251171,Heartbleed
447713,BENIGN
21152,DoS Slowhttptest
27597,DoS slowloris
229878,BENIGN
21654,BENIGN
313937,DoS GoldenEye
12829,BENIGN
120307,DoS Hulk
